# MS COCO Multi-Label Classification Challenge

The objective is to predict, for each image, all the object categories that are present among the 80 COCO classes.

I progressively compared several transfer-learning strategies and architectures, including ResNet50, ConvNeXt-Small, ViT-B/16 and ConvNeXt-Base with ML-Decoder. I also investigated different loss functions, image transformations, test-time augmentation, threshold calibration and probability ensembling.

My best hidden-test result before the ML-Decoder experiment was **F1 = 0.7018**, obtained with an ensemble of ConvNeXt-Small, ViT-B/16 and ResNet50 Focal.

In parallel, my teammate explored ViT-Large and higher input resolutions. Increasing the image resolution from 224×224 to 384×384 gave her **0.8045 F1 on validation** and **0.7255 on the hidden test**. 

## 1. Experimental setup

In [1]:
import sys
import json
import random
import math
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image, ImageOps

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from tqdm.auto import tqdm

import timm


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type != "cuda":
    raise RuntimeError("CUDA is required for the experiments in this notebook.")

torch.backends.cudnn.benchmark = True


PROJECT_ROOT = Path.home() / "coco_project"
DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)


print("Python :", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda)
print("GPU    :", torch.cuda.get_device_name(0))
print(
    "VRAM   :",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB",
)

Python : /home/lenovo-loq/venvs/coco/bin/python
PyTorch: 2.11.0+cu130
CUDA   : 13.0
GPU    : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM   : 7.62 GB


All experiments were run locally on my laptop using an NVIDIA RTX 4060 Laptop GPU with 8 GB of VRAM. I used the same local environment and the same train/validation split throughout the study so that model comparisons remain consistent.

In [2]:
DATA_ROOT = DATA_DIR / "ms-coco"

TRAIN_IMAGES = DATA_ROOT / "images" / "train"
TEST_IMAGES = DATA_ROOT / "images" / "test"
TRAIN_LABELS = DATA_ROOT / "labels" / "train"

train_images = sorted(TRAIN_IMAGES.glob("*.jpg"))
test_images = sorted(TEST_IMAGES.glob("*.jpg"))
label_files = sorted(TRAIN_LABELS.glob("*.cls"))

assert len(train_images) == 65000
assert len(test_images) == 4952
assert len(label_files) == 65000

print("Train images :", len(train_images))
print("Test images  :", len(test_images))
print("Train labels :", len(label_files))

Train images : 65000
Test images  : 4952
Train labels : 65000


## 2. Multi-label targets and train/validation split

Each image can contain several object categories, so I represent every annotation as an 80-dimensional multi-hot vector.

For all my experiments, I use the same **80/20 iterative multi-label stratified split** with a fixed seed (`42`). Unlike a standard stratified split, this method is adapted to multi-label data and helps preserve the frequency of rare classes in both subsets.

I keep this split fixed throughout the study so that differences between experiments come from the models and training strategies rather than from different validation samples. The test set is never used for model selection.

In [3]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit


CLASSES = (
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat", "traffic light",
    "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep", "cow",
    "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee",
    "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove", "skateboard", "surfboard",
    "tennis racket", "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
    "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch",
    "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse", "remote", "keyboard",
    "cell phone", "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase",
    "scissors", "teddy bear", "hair drier", "toothbrush",
)

NUM_CLASSES = len(CLASSES)

image_ids = [path.stem for path in label_files]

targets = np.zeros(
    (len(label_files), NUM_CLASSES),
    dtype=np.uint8,
)

for i, label_file in enumerate(label_files):
    labels = [int(x) for x in label_file.read_text().split()]
    targets[i, labels] = 1


SPLIT_FILE = CHECKPOINT_DIR / "multilabel_split_seed42.npz"

if SPLIT_FILE.exists():
    split = np.load(SPLIT_FILE)
    train_indices = split["train_indices"]
    val_indices = split["val_indices"]

else:
    splitter = MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=SEED,
    )

    train_indices, val_indices = next(
        splitter.split(
            np.zeros(len(targets)),
            targets,
        )
    )

    np.savez(
        SPLIT_FILE,
        train_indices=train_indices,
        val_indices=val_indices,
    )


assert len(set(train_indices) & set(val_indices)) == 0
assert len(train_indices) + len(val_indices) == len(targets)

print("Target matrix :", targets.shape)
print("Train samples :", len(train_indices))
print("Validation    :", len(val_indices))
print(
    "Labels/image  :",
    round(targets.sum(axis=1).mean(), 3),
)

Target matrix : (65000, 80)
Train samples : 52023
Validation    : 12977
Labels/image  : 2.926


### Class imbalance

The label distribution is strongly imbalanced. Some categories appear in tens of thousands of images, while others occur only a few hundred or even around one hundred times.

I therefore keep track of class frequencies throughout the experiments, since rare categories play an important role both during training and in the official evaluation metric.

In [4]:
class_counts = targets.sum(axis=0)

class_distribution = pd.DataFrame({
    "class": CLASSES,
    "positive_samples": class_counts,
    "frequency_%": 100 * class_counts / len(targets),
})

print("Most frequent classes")
display(
    class_distribution
    .sort_values("positive_samples", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

print("Rarest classes")
display(
    class_distribution
    .sort_values("positive_samples")
    .head(10)
    .reset_index(drop=True)
)

Most frequent classes


,class,positive_samples,frequency_%
0,person,35494,54.606154
1,chair,7043,10.835385
2,car,6811,10.478462
3,dining table,6546,10.070769
4,cup,5057,7.780000
5,bottle,4719,7.260000
6,bowl,3960,6.092308
7,handbag,3826,5.886154
8,truck,3416,5.255385
9,backpack,3141,4.832308


Rarest classes


,class,positive_samples,frequency_%
0,hair drier,102,0.156923
1,toaster,117,0.180000
2,parking meter,396,0.609231
3,bear,516,0.793846
4,scissors,555,0.853846
5,toothbrush,561,0.863077
6,hot dog,667,1.026154
7,sheep,809,1.244615
8,microwave,832,1.280000
9,donut,856,1.316923


## 3. Input pipeline and image transformations

During my first experiments, I used the standard ImageNet preprocessing based on resizing followed by a center crop.

For a multi-label problem, however, aggressive cropping can remove an object while its label remains positive. I therefore also evaluated a **letterbox transformation**, which preserves the complete image by resizing it while keeping its aspect ratio and padding the remaining area.

The two views later proved complementary in my ensemble experiments. For the ML-Decoder model, I use the full-frame letterbox representation during both training and validation.

In [5]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

NUM_WORKERS = 2


class COCOMultiLabelDataset(Dataset):
    def __init__(self, image_dir, image_ids, targets, indices, transform=None):
        self.image_dir = Path(image_dir)
        self.image_ids = image_ids
        self.targets = targets
        self.indices = np.asarray(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        source_idx = self.indices[idx]
        image_id = self.image_ids[source_idx]

        image = Image.open(
            self.image_dir / f"{image_id}.jpg"
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        target = torch.tensor(
            self.targets[source_idx],
            dtype=torch.float32,
        )

        return image, target


class COCOTestDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_paths = sorted(Path(image_dir).glob("*.jpg"))
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, image_path.stem


class Letterbox224:
    def __init__(self, size=224):
        self.size = size

    def __call__(self, image):
        return ImageOps.pad(
            image,
            (self.size, self.size),
            method=Image.Resampling.BICUBIC,
            color=(124, 116, 104),
        )


crop_transform_224 = transforms.Compose([
    transforms.Resize(
        256,
        interpolation=InterpolationMode.BICUBIC,
    ),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


letterbox_transform_224 = transforms.Compose([
    Letterbox224(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


train_transform_mld = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.12,
            contrast=0.12,
            saturation=0.08,
            hue=0.02,
        )
    ], p=0.4),

    Letterbox224(224),

    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


val_dataset_mld = COCOMultiLabelDataset(
    TRAIN_IMAGES,
    image_ids,
    targets,
    val_indices,
    transform=letterbox_transform_224,
)

val_loader_mld = DataLoader(
    val_dataset_mld,
    batch_size=8,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)

print("Validation samples:", len(val_dataset_mld))

Validation samples: 12977


## 4. Evaluation protocol

The challenge metric differs from the standard macro-F1.

Precision and recall are first computed independently for every class. They are then aggregated using **normalized inverse class-frequency weights**, before computing their harmonic mean. As a consequence, errors on rare classes can have a much larger impact than errors on frequent categories.

The hidden-test class frequencies are not available during development. For local model comparison, I therefore use the inverse frequencies of the full 65,000-image annotated training set as a stable proxy for the official weights.

I also found that directly optimizing all 80 thresholds on the complete validation set produced overly optimistic local scores that did not transfer well to the hidden test. I therefore use **5-fold out-of-fold threshold calibration** for model selection. Thresholds are fitted on one part of the validation data and evaluated on unseen folds.

For exact consistency with the challenge implementation, predictions use the strict comparison `probability > threshold`.

In [6]:
full_class_counts = torch.tensor(
    targets.sum(axis=0),
    dtype=torch.float32,
)

proxy_weights = 1.0 / full_class_counts
proxy_weights /= proxy_weights.sum()

In [7]:
def calibrate_class_thresholds_strict(
    probabilities,
    targets_tensor,
    min_threshold=0.05,
    max_threshold=0.80,
    step=0.01,
):
    grid = torch.arange(
        min_threshold,
        max_threshold + step / 2,
        step,
    )

    y = targets_tensor.bool()

    best_thresholds = torch.full(
        (NUM_CLASSES,),
        0.5,
        dtype=torch.float32,
    )

    best_f1 = torch.zeros(
        NUM_CLASSES,
        dtype=torch.float32,
    )

    for c in range(NUM_CLASSES):
        for threshold in grid:
            pred = probabilities[:, c] > threshold

            tp = (pred & y[:, c]).sum().float()
            fp = (pred & ~y[:, c]).sum().float()
            fn = (~pred & y[:, c]).sum().float()

            score = (
                2 * tp
                / (2 * tp + fp + fn).clamp_min(1)
            )

            if score > best_f1[c]:
                best_f1[c] = score
                best_thresholds[c] = threshold

    return best_thresholds, best_f1


def official_metric_fixed_weights(
    predictions,
    targets_tensor,
    weights,
):
    pred = predictions.bool()
    y = targets_tensor.bool()

    tp = (pred & y).sum(dim=0).float()
    fp = (pred & ~y).sum(dim=0).float()
    fn = (~pred & y).sum(dim=0).float()

    precision = torch.where(
        tp > 0,
        tp / (tp + fp),
        torch.zeros_like(tp),
    )

    recall = torch.where(
        tp > 0,
        tp / (tp + fn),
        torch.zeros_like(tp),
    )

    P = (precision * weights).sum()
    R = (recall * weights).sum()

    F1 = 2 * P * R / (P + R + 1e-12)

    return {
        "f1": F1.item(),
        "precision": P.item(),
        "recall": R.item(),
    }


def standard_macro_metrics(
    probabilities,
    targets_tensor,
    thresholds,
):
    pred = probabilities > thresholds.unsqueeze(0)
    y = targets_tensor.bool()

    tp = (pred & y).sum(dim=0).float()
    fp = (pred & ~y).sum(dim=0).float()
    fn = (~pred & y).sum(dim=0).float()

    precision = tp / (tp + fp).clamp_min(1)
    recall = tp / (tp + fn).clamp_min(1)
    f1 = 2 * tp / (2 * tp + fp + fn).clamp_min(1)

    return {
        "macro_f1": f1.mean().item(),
        "macro_precision": precision.mean().item(),
        "macro_recall": recall.mean().item(),
    }


## 5. Experimental progression

I did not start directly with the final ensemble or ML-Decoder. I progressively tested different architectures, loss functions and inference strategies, using the same train/validation split throughout the experiments.

The objective of this exploratory phase was to identify three things:

- which architecture gives the strongest individual representation;
- how class imbalance affects the predictions;
- whether models with different inductive biases can be combined to improve generalization.

The following sections summarize the main experiments that influenced my final approach. I keep only the relevant results here rather than repeating all the training code.

### 5.1 ResNet50 baseline and partial fine-tuning

I first used a ResNet50 pretrained on ImageNet as a simple convolutional baseline.

With the backbone frozen and only the classification layer trained, I obtained a validation Macro-F1 of **0.6144**.

I then fine-tuned the last residual stage (`layer4`) together with the classification head. This increased the Macro-F1 to **0.6771**, showing that adapting the high-level visual features to the multi-label task was significantly more effective than using the pretrained backbone only as a fixed feature extractor.

After class-specific threshold calibration, the validation Macro-F1 reached **0.6969**.

This experiment established partial fine-tuning as a better starting point than a fully frozen backbone.

### 5.2 Loss functions and class imbalance

Because the dataset is strongly imbalanced, I tested alternatives to the standard binary cross-entropy loss.

Starting from the fine-tuned ResNet50, I compared:

| Loss | Validation Macro-F1 @ 0.5 | Observation |
|---|---:|---|
| BCE | **0.6771** | Best balanced baseline |
| Weighted BCE | 0.6634 | Increased recall but produced too many false positives |
| Focal Loss | 0.6699 | Slightly below standard BCE |

Weighted BCE reached **0.6885** after threshold calibration, but it still remained below the calibrated standard BCE result of **0.6969**.

These experiments showed me that explicitly increasing the contribution of rare positive labels does not automatically improve the final prediction quality. In particular, aggressive class weighting tends to trade too much precision for recall.

I therefore kept standard BCE for the next architecture comparisons and treated loss-function design separately from model capacity.

### 5.3 Moving to ConvNeXt

The ResNet experiments suggested that the main limitation was no longer only the loss function. I therefore moved to a stronger convolutional architecture, **ConvNeXt-Small**, pretrained on ImageNet.

After full fine-tuning, the model reached:

- **0.7350 Macro-F1** with the default threshold of 0.5;
- **0.7533 Macro-F1** after class-specific threshold calibration;
- **0.7540 Macro-F1** with horizontal-flip test-time augmentation and calibrated thresholds.

The improvement over ResNet50 was substantial. Horizontal-flip TTA provided only a small additional gain, but I kept it because its inference cost was low.

At this stage, changing the architecture produced a much larger improvement than changing the loss function.

### 5.4 Vision Transformer

I then trained a complementary transformer architecture, **ViT-B/16**, pretrained on ImageNet-21K and fine-tuned at 224×224.

Because of the 8 GB GPU memory constraint, I used mixed precision, gradient accumulation and gradient checkpointing. I also used layer-wise learning-rate decay and an exponential moving average of the model weights.

The best checkpoint was obtained at epoch 9:

- **0.7542 Macro-F1** at threshold 0.5;
- **0.7708 Macro-F1** after threshold calibration;
- **0.7718 Macro-F1** with horizontal-flip TTA and calibration.

The ViT outperformed ConvNeXt individually, while producing sufficiently different predictions to make probability ensembling worthwhile.

### 5.5 Probability ensembling

Since ConvNeXt and ViT rely on different visual representations, I combined their predicted probabilities instead of selecting only the best individual model.

The best validation mixture using horizontal-flip TTA was:

- **40% ConvNeXt-Small**
- **60% ViT-B/16**

With class-specific threshold calibration, this ensemble reached a standard validation Macro-F1 of **0.7821**, higher than either model individually.

This confirmed that architectural diversity was useful: even though ViT was the stronger individual model, ConvNeXt still contributed complementary information.

### 5.6 Threshold calibration and validation–test gap

At first, I optimized one decision threshold per class directly on the complete validation set. This substantially improved local scores.

However, after comparing several submissions with the hidden test set, I observed that aggressive threshold optimization did not generalize reliably. In particular, thresholds optimized directly for the official inverse-frequency metric could produce very strong validation results while decreasing the hidden-test score.

I therefore changed the evaluation protocol and used out-of-fold calibration for model selection. This gave a more conservative estimate of generalization and reduced the risk of selecting a model only because it overfitted the validation thresholds.

This validation–test gap became one of the main difficulties of the project.

### 5.7 Full-frame representation and multi-view inference

During the error analysis, I noticed an important limitation of the standard center-crop preprocessing.

Many original images are rectangular. Resizing the short side and then applying a 224×224 center crop can remove objects located close to the image borders. In a multi-label setting, the corresponding label remains positive even when the object has disappeared from the model input.

I therefore introduced the letterbox transformation described earlier to preserve the complete image.

Interestingly, neither crop nor letterbox was universally better. Their predictions were complementary. Combining both views produced a stronger out-of-fold official proxy score than using either view alone.

The selected multi-view configuration combined:

- a crop view based on **60% ConvNeXt / 40% ViT**;
- a letterbox view based on **30% ConvNeXt / 70% ViT**;
- **30% crop / 70% letterbox** for the final multi-view prediction.

This improved the local out-of-fold official proxy to approximately **0.7242**.

### 5.8 Adding ResNet as a diversity model

Although ResNet50 was weaker than ConvNeXt and ViT when evaluated alone, I tested whether it could still improve the ensemble by providing different errors.

Among the ResNet variants, the Focal-Loss model provided the most useful complementary predictions. Adding it to the ConvNeXt–ViT multi-view ensemble improved the out-of-fold official proxy from approximately **0.7242 to 0.7297**.

The final mixture was:

- **70% ConvNeXt–ViT multi-view ensemble**
- **30% ResNet50 Focal**

This model achieved my best hidden-test result at this stage:

**F1 = 0.7018**

with:

- Precision: **0.7847**
- Recall: **0.6347**

An important conclusion from this experiment is that the strongest individual model is not necessarily the most useful ensemble member. Prediction diversity can be valuable even when the standalone score is lower.

### 5.9 Additional experiments that did not improve the final model

I also investigated several alternatives that were not retained in the final pipeline.

**Top-K decoding.**  
I estimated the expected number of positive predictions per class from the training distribution and used this prior instead of probability thresholds. The best out-of-fold official proxy was approximately **0.7013**, below the threshold-based multi-view ensemble.

**Rare-class specialist.**  
I fine-tuned a ViT specifically on difficult rare categories such as `toaster`, `hair drier`, `scissors`, `toothbrush` and `parking meter`. The specialist did not improve the average precision of these classes compared with the original ViT and was therefore discarded.

**ViT full-frame adaptation.**  
I adapted the best ViT checkpoint specifically to letterbox inputs. Performance peaked after the first epoch and then decreased, without improving the rare-class representation.

These unsuccessful experiments were still useful because they suggested that the main limitation could not be solved simply by increasing rare-class weights or by changing the decoding strategy.

### 5.10 Complementary experiments by my teammate

In parallel with my experiments, my teammate focused on the Vision Transformer branch.

She tested **ViT-Large** and investigated the effect of increasing the input resolution. Moving from 224×224 to **384×384** improved her validation F1 to **0.8045**.

Her corresponding hidden-test score was **0.7255**, which is currently higher than my best hidden-test result of 0.7018, although a noticeable validation–test gap remains.

Our current directions are therefore complementary:

- my experiments focus on architecture diversity, multi-view inference, rare-class behaviour and ensemble design;
- her experiments focus on ViT-Large, higher image resolution and improving the generalization of the stronger individual transformer model.

These two branches can later be compared or combined if their prediction errors are sufficiently complementary.

### 5.11 Motivation for ML-Decoder

The previous experiments showed two persistent limitations.

First, small and rare categories remained considerably harder than visually distinctive categories such as `bear`, `zebra` or `giraffe`.

Second, increasing the complexity of threshold calibration improved validation scores more easily than hidden-test performance.

For my next experiment, I therefore decided to change the representation itself rather than further optimize the decision thresholds.

I selected **ConvNeXt-Base + ML-Decoder with Asymmetric Loss**.

The motivation was to combine:

- a stronger ConvNeXt backbone;
- a classification head designed specifically for multi-label recognition;
- spatial feature information rather than relying only on a conventional global classification head;
- Asymmetric Loss to reduce the influence of easy negative labels in this highly imbalanced problem.

The next section describes this experiment in detail.

## 6. ConvNeXt-Base + ML-Decoder + Asymmetric Loss

After the previous experiments, I wanted to test an architecture designed more specifically for multi-label classification.

I used a **ConvNeXt-Base** backbone pretrained on ImageNet and replaced its standard classification head with **ML-Decoder**.

Unlike a conventional global classification head, ML-Decoder operates on the spatial features produced by the backbone and uses class-related queries to generate the final predictions. I expected this representation to be more suitable for images containing several objects, especially when some of them occupy only a small part of the image.

I trained this model with **Asymmetric Loss (ASL)** instead of BCE. ASL reduces the contribution of easy negative labels, which are very numerous in this dataset, while preserving the learning signal from positive labels.

For this experiment, I used the full-frame letterbox representation rather than center cropping.

In [8]:
ML_DECODER_ROOT = PROJECT_ROOT / "external" / "ML_Decoder"
sys.path.insert(0, str(ML_DECODER_ROOT))

from src_files.ml_decoder.ml_decoder import (
    add_ml_decoder_head,
    TransformerDecoderLayerOptimal,
)

from src_files.loss_functions.losses import (
    AsymmetricLossOptimized,
)

> **Implementation note.**  
> The original ML-Decoder implementation targets an older PyTorch Transformer API. Since my local environment uses PyTorch 2.11, I apply the following compatibility patch without modifying the ML-Decoder computation itself.

In [9]:
if not hasattr(TransformerDecoderLayerOptimal, "self_attn"):
    TransformerDecoderLayerOptimal.self_attn = property(
        lambda self: self.multihead_attn
    )

_original_mld_forward = TransformerDecoderLayerOptimal.forward


def _mld_forward_compat(
    self,
    tgt,
    memory,
    tgt_mask=None,
    memory_mask=None,
    tgt_key_padding_mask=None,
    memory_key_padding_mask=None,
    tgt_is_causal=False,
    memory_is_causal=False,
):
    return _original_mld_forward(
        self,
        tgt,
        memory,
        tgt_mask=tgt_mask,
        memory_mask=memory_mask,
        tgt_key_padding_mask=tgt_key_padding_mask,
        memory_key_padding_mask=memory_key_padding_mask,
    )


TransformerDecoderLayerOptimal.forward = _mld_forward_compat

### 6.1 Model architecture

I used `convnext_base.fb_in22k_ft_in1k` as the backbone and attached an ML-Decoder head producing the 80 challenge labels.

The resulting network contains approximately **94 million parameters**. The backbone starts from pretrained ImageNet weights, while the multi-label decoder is adapted to the target task.

In [10]:
MLD_MODEL_NAME = "convnext_base.fb_in22k_ft_in1k"

model_mld = timm.create_model(
    MLD_MODEL_NAME,
    pretrained=False,
    num_classes=NUM_CLASSES,
    drop_path_rate=0.1,
)

model_mld = add_ml_decoder_head(
    model_mld,
    num_classes=NUM_CLASSES,
    num_of_groups=NUM_CLASSES,
    decoder_embedding=768,
)

model_mld = model_mld.to(device)

total_params = sum(
    p.numel()
    for p in model_mld.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model_mld.parameters()
    if p.requires_grad
)

print(f"Parameters : {total_params / 1e6:.1f} M")
print(f"Trainable  : {trainable_params / 1e6:.1f} M")

Parameters : 94.0 M
Trainable  : 93.9 M


The original training run initialized the ConvNeXt backbone from pretrained ImageNet weights. In this cleaned notebook I reconstruct the architecture without downloading the pretrained weights again, because the trained checkpoint is loaded below.

### 6.2 Training strategy

I trained the model for 10 epochs using the same 52,023 training images defined by the fixed split.

The main training configuration was:

| Parameter | Value |
|---|---:|
| Input size | 224×224 |
| Training representation | Letterbox |
| Loss | Asymmetric Loss |
| `gamma_neg` | 4 |
| `gamma_pos` | 0 |
| ASL clipping | 0.05 |
| Optimizer | AdamW |
| Backbone maximum LR | 1e-5 |
| ML-Decoder maximum LR | 1e-4 |
| Weight decay | 0.05 |
| Micro-batch size | 4 |
| Gradient accumulation | 8 |
| Effective batch size | 32 |
| Scheduler | OneCycleLR |
| Epochs | 10 |
| Gradient clipping | 1.0 |

I used mixed-precision training to reduce memory consumption on the RTX 4060.

> ⚠️ **Training cell — not rerun in the final notebook.**  
> The experiment has already been completed and the best checkpoint is stored locally. The code is kept here for reproducibility.

### 6.3 Training results

I monitored both Macro AP and an inverse-frequency weighted AP during training. I used weighted AP for checkpoint selection because it is threshold-independent and gives more importance to the rare classes that strongly affect the challenge metric.

The best weighted AP was reached at epoch 4.

| Epoch | Train loss | Weighted AP | Macro AP |
|---:|---:|---:|---:|
| 1 | 2.1208 | 0.6938 | 0.8099 |
| 2 | 1.3934 | 0.7327 | 0.8300 |
| 3 | 1.1980 | 0.7476 | 0.8366 |
| **4** | **1.0551** | **0.7562** | **0.8386** |
| 5 | 0.9288 | 0.7531 | 0.8375 |
| 6 | 0.8126 | 0.7513 | 0.8362 |
| 7 | 0.7136 | 0.7507 | 0.8343 |
| 8 | 0.6306 | 0.7501 | 0.8327 |
| 9 | 0.5796 | 0.7492 | 0.8325 |
| 10 | 0.5551 | 0.7495 | 0.8324 |

Training loss continued to decrease after epoch 4, while both weighted AP and Macro AP slightly deteriorated. I therefore retained the epoch-4 checkpoint instead of the final epoch.

This was also preferable to selecting the checkpoint from F1 at a fixed threshold of 0.5, since the final decision thresholds are calibrated separately later.

### 6.4 Selected checkpoint

I reload the epoch-4 checkpoint selected from weighted AP. The backbone had originally been initialized from ImageNet-pretrained weights during training; here I reconstruct the architecture with `pretrained=False` because the complete trained state is already stored in the checkpoint.

In [11]:
MLD_CHECKPOINT = (
    CHECKPOINT_DIR
    / "best_convnext_base_mldecoder_asl.pt"
)

checkpoint_mld = torch.load(
    MLD_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

model_mld.load_state_dict(
    checkpoint_mld["model_state_dict"]
)

model_mld = model_mld.to(device)
model_mld.eval()

print("Selected epoch :", checkpoint_mld["epoch"])
print("Weighted AP    :", round(checkpoint_mld["weighted_ap"], 4))
print("Macro AP       :", round(checkpoint_mld["macro_ap"], 4))

Selected epoch : 4
Weighted AP    : 0.7562
Macro AP       : 0.8386


### 6.5 Test-time augmentation

For inference, I use horizontal-flip test-time augmentation. I evaluate each image in its original orientation and after horizontal flipping, then average the two probability vectors.

I had already used the same strategy with ConvNeXt and ViT. Its improvement was small but consistent, and it adds no additional trainable parameters.

In [12]:
@torch.inference_mode()
def predict_validation_tta(model, loader):
    model.eval()

    all_probs = []
    all_targets = []

    for images, batch_targets in tqdm(
        loader,
        desc="ML-Decoder validation",
    ):
        images = images.to(
            device,
            non_blocking=True,
        )

        flipped = torch.flip(
            images,
            dims=[3],
        )

        with torch.amp.autocast(
            "cuda",
            enabled=device.type == "cuda",
        ):
            logits = model(images)
            logits_flip = model(flipped)

        probs = (
            torch.sigmoid(logits.float())
            + torch.sigmoid(logits_flip.float())
        ) / 2

        all_probs.append(probs.cpu())
        all_targets.append(batch_targets.cpu())

    return (
        torch.cat(all_probs),
        torch.cat(all_targets),
    )

In [13]:
MLD_VAL_CACHE = (
    CHECKPOINT_DIR
    / "convnext_base_mldecoder_asl_tta_val_probs.pt"
)

if MLD_VAL_CACHE.exists():
    saved = torch.load(
        MLD_VAL_CACHE,
        map_location="cpu",
        weights_only=False,
    )

    mld_tta_probs = saved["probs"]
    mld_targets = saved["targets"]

else:
    mld_tta_probs, mld_targets = (
        predict_validation_tta(
            model_mld,
            val_loader_mld,
        )
    )

    torch.save(
        {
            "probs": mld_tta_probs,
            "targets": mld_targets,
        },
        MLD_VAL_CACHE,
    )

assert mld_tta_probs.shape == (
    len(val_indices),
    NUM_CLASSES,
)

print("Validation probabilities:", mld_tta_probs.shape)

Validation probabilities: torch.Size([12977, 80])


### 6.6 Out-of-fold evaluation

The checkpoint was selected using a threshold-free metric, but the challenge ultimately evaluates binary predictions.

To avoid selecting class thresholds and evaluating them on the same validation examples, I use 5-fold multi-label stratified out-of-fold calibration. For each fold, I estimate the 80 class thresholds on the other four folds and evaluate them only on the held-out samples.

This produces a more conservative estimate than calibrating and evaluating thresholds on the complete validation set.

In [16]:
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold


cv = MultilabelStratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

folds = list(
    cv.split(
        np.zeros((len(mld_targets), 1)),
        mld_targets.numpy(),
    )
)


def evaluate_probs_oof(probabilities, targets_tensor, folds):
    oof_predictions = torch.zeros_like(
        targets_tensor,
        dtype=torch.bool,
    )

    fold_scores = []

    for calibration_idx, evaluation_idx in folds:
        calibration_idx = torch.as_tensor(calibration_idx)
        evaluation_idx = torch.as_tensor(evaluation_idx)

        thresholds, _ = calibrate_class_thresholds_strict(
            probabilities[calibration_idx],
            targets_tensor[calibration_idx],
        )

        fold_predictions = (
            probabilities[evaluation_idx]
            > thresholds.unsqueeze(0)
        )

        oof_predictions[evaluation_idx] = fold_predictions

        fold_metrics = official_metric_fixed_weights(
            fold_predictions,
            targets_tensor[evaluation_idx],
            proxy_weights,
        )

        fold_scores.append(fold_metrics["f1"])

    overall_metrics = official_metric_fixed_weights(
        oof_predictions,
        targets_tensor,
        proxy_weights,
    )

    return {
        **overall_metrics,
        "fold_mean": float(np.mean(fold_scores)),
        "fold_std": float(np.std(fold_scores)),
        "fold_min": float(np.min(fold_scores)),
        "oof_predictions": oof_predictions,
    }


mld_oof = evaluate_probs_oof(
    mld_tta_probs,
    mld_targets,
    folds,
)

mld_oof_summary = pd.DataFrame({
    "Metric": [
        "Official proxy F1",
        "Precision",
        "Recall",
        "Fold mean",
        "Fold standard deviation",
        "Worst fold",
    ],
    "Value": [
        mld_oof["f1"],
        mld_oof["precision"],
        mld_oof["recall"],
        mld_oof["fold_mean"],
        mld_oof["fold_std"],
        mld_oof["fold_min"],
    ],
})

display(
    mld_oof_summary.style
    .format({"Value": "{:.4f}"})
    .hide(axis="index")
)

Metric,Value
Official proxy F1,0.7145
Precision,0.7312
Recall,0.6985
Fold mean,0.7134
Fold standard deviation,0.0271
Worst fold,0.6745


## 7. Reference ensemble

Before introducing ML-Decoder, my strongest model was an ensemble combining three architectures — **ConvNeXt-Small, ViT-B/16 and ResNet50 Focal** — evaluated with two complementary image representations: **center crop** and **letterbox**.

I first combine ConvNeXt and ViT predictions separately for each image representation:

$$
P_{\text{crop}}
=
0.60 \, P_{\text{ConvNeXt,crop}}
+
0.40 \, P_{\text{ViT,crop}}
$$

$$
P_{\text{letterbox}}
=
0.30 \, P_{\text{ConvNeXt,letterbox}}
+
0.70 \, P_{\text{ViT,letterbox}}
$$

These two views are then combined into a multi-view prediction:

$$
P_{\text{multi-view}}
=
0.30 \, P_{\text{crop}}
+
0.70 \, P_{\text{letterbox}}
$$

For ResNet50 Focal, I also combine crop and letterbox predictions:

$$
P_{\text{ResNet}}
=
0.60 \, P_{\text{ResNet,crop}}
+
0.40 \, P_{\text{ResNet,letterbox}}
$$

Finally, the reference ensemble is defined as:

$$
P_{\text{baseline}}
=
0.70 \, P_{\text{multi-view}}
+
0.30 \, P_{\text{ResNet}}
$$

This configuration reached an out-of-fold official proxy F1 of approximately **0.7297** and a hidden-test F1 of **0.7018**.

I use this ensemble as the reference baseline to evaluate whether ML-Decoder contributes complementary information and improves the final predictions.

In [17]:
BASELINE_VAL_CACHE = (
    CHECKPOINT_DIR
    / "baseline_3model_val_probs.pt"
)

baseline_saved = torch.load(
    BASELINE_VAL_CACHE,
    map_location="cpu",
    weights_only=False,
)

baseline_val_probs = baseline_saved["probs"]
baseline_targets = baseline_saved["targets"]

assert baseline_val_probs.shape == mld_tta_probs.shape
assert torch.equal(
    baseline_targets,
    mld_targets,
)

print(
    "Validation probabilities:",
    baseline_val_probs.shape,
)

Validation probabilities: torch.Size([12977, 80])


### 7.1 Reference out-of-fold performance

I first reevaluate the saved baseline probabilities using exactly the same out-of-fold procedure used for ML-Decoder. This ensures that both models are compared under the same calibration protocol.

In [18]:
baseline_oof = evaluate_probs_oof(
    baseline_val_probs,
    baseline_targets,
    folds,
)

baseline_oof_summary = pd.DataFrame({
    "Metric": [
        "Official proxy F1",
        "Precision",
        "Recall",
        "Fold mean",
        "Fold standard deviation",
        "Worst fold",
    ],
    "Value": [
        baseline_oof["f1"],
        baseline_oof["precision"],
        baseline_oof["recall"],
        baseline_oof["fold_mean"],
        baseline_oof["fold_std"],
        baseline_oof["fold_min"],
    ],
})

display(
    baseline_oof_summary.style
    .format({"Value": "{:.4f}"})
    .hide(axis="index")
)

Metric,Value
Official proxy F1,0.7297
Precision,0.7762
Recall,0.6885
Fold mean,0.7282
Fold standard deviation,0.0102
Worst fold,0.7121


The reference ensemble remains stronger than ML-Decoder alone:

- reference ensemble: **0.7297 OOF F1**
- ML-Decoder: **0.7145 OOF F1**

However, ML-Decoder has a different architecture, loss function and spatial decoding mechanism. I therefore test whether combining the two probability distributions improves performance even though ML-Decoder is weaker as a standalone model.

## 8. ML-Decoder as an ensemble component

I combine the reference ensemble and ML-Decoder at the probability level:


In [19]:
ensemble_results = []

for mld_weight in np.arange(
    0.0,
    0.51,
    0.05,
):
    baseline_weight = 1.0 - mld_weight

    ensemble_probs = (
        baseline_weight * baseline_val_probs
        + mld_weight * mld_tta_probs
    )

    metrics = evaluate_probs_oof(
        ensemble_probs,
        mld_targets,
        folds,
    )

    ensemble_results.append({
        "ML-Decoder weight": mld_weight,
        "Baseline weight": baseline_weight,
        "OOF F1": metrics["f1"],
        "Precision": metrics["precision"],
        "Recall": metrics["recall"],
        "Fold std": metrics["fold_std"],
        "Worst fold": metrics["fold_min"],
    })


ensemble_scan_df = (
    pd.DataFrame(ensemble_results)
    .sort_values(
        "OOF F1",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    ensemble_scan_df.style
    .format({
        "ML-Decoder weight": "{:.2f}",
        "Baseline weight": "{:.2f}",
        "OOF F1": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "Fold std": "{:.4f}",
        "Worst fold": "{:.4f}",
    })
    .hide(axis="index")
)

ML-Decoder weight,Baseline weight,OOF F1,Precision,Recall,Fold std,Worst fold
0.40,0.60,0.7500,0.8180,0.6923,0.0349,0.6817
0.45,0.55,0.7492,0.8166,0.6920,0.0382,0.6831
0.50,0.50,0.7473,0.8088,0.6945,0.0384,0.6844
0.30,0.70,0.7445,0.8053,0.6922,0.0330,0.6812
0.15,0.85,0.7419,0.7859,0.7025,0.0213,0.7126
0.25,0.75,0.7409,0.7822,0.7037,0.0277,0.7023
0.20,0.80,0.7378,0.7763,0.7029,0.0275,0.7001
0.35,0.65,0.7338,0.7871,0.6873,0.0296,0.6819
0.00,1.00,0.7297,0.7762,0.6885,0.0102,0.7121
0.05,0.95,0.7196,0.7730,0.6731,0.0102,0.7120


In [20]:
SELECTED_MLD_WEIGHT = 0.40
SELECTED_BASELINE_WEIGHT = 0.60

selected_row = ensemble_scan_df[
    np.isclose(
        ensemble_scan_df["ML-Decoder weight"],
        SELECTED_MLD_WEIGHT,
    )
].iloc[0]

selected_comparison = pd.DataFrame({
    "Model": [
        "Reference ensemble",
        "60% reference + 40% ML-Decoder",
    ],
    "OOF F1": [
        baseline_oof["f1"],
        selected_row["OOF F1"],
    ],
    "Precision": [
        baseline_oof["precision"],
        selected_row["Precision"],
    ],
    "Recall": [
        baseline_oof["recall"],
        selected_row["Recall"],
    ],
    "Fold std": [
        baseline_oof["fold_std"],
        selected_row["Fold std"],
    ],
    "Worst fold": [
        baseline_oof["fold_min"],
        selected_row["Worst fold"],
    ],
})

display(
    selected_comparison.style
    .format({
        "OOF F1": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "Fold std": "{:.4f}",
        "Worst fold": "{:.4f}",
    })
    .hide(axis="index")
)

Model,OOF F1,Precision,Recall,Fold std,Worst fold
Reference ensemble,0.7297,0.7762,0.6885,0.0102,0.7121
60% reference + 40% ML-Decoder,0.7500,0.8180,0.6923,0.0349,0.6817


The 40% ML-Decoder mixture increases the aggregate OOF proxy F1 from **0.7297 to 0.7500**, an absolute improvement of approximately **0.0202**.

The gain mainly comes from higher precision, while recall changes only slightly.

However, the fold-level behaviour is less reassuring. The standard deviation increases from approximately **0.0102 to 0.0349**, and the worst fold decreases from **0.7121 to 0.6817**.

I therefore interpret the higher aggregate OOF score cautiously. The ML-Decoder predictions appear complementary, but the improvement is substantially less stable across validation subsets than the reference ensemble.

### 8.1 Full-validation calibration

After selecting the mixture weight using out-of-fold evaluation, I calibrate one threshold per class on the complete validation set. These thresholds are used only to produce the final hidden-test submission.

The resulting full-validation score is reported for completeness, but I do not use it as the main generalization estimate because the same validation samples are used for threshold calibration and evaluation.

In [21]:
selected_val_probs = (
    SELECTED_BASELINE_WEIGHT
    * baseline_val_probs
    + SELECTED_MLD_WEIGHT
    * mld_tta_probs
)

selected_thresholds, _ = (
    calibrate_class_thresholds_strict(
        selected_val_probs,
        mld_targets,
    )
)

selected_val_predictions = (
    selected_val_probs
    > selected_thresholds.unsqueeze(0)
)

selected_proxy = (
    official_metric_fixed_weights(
        selected_val_predictions,
        mld_targets,
        proxy_weights,
    )
)

selected_standard = standard_macro_metrics(
    selected_val_probs,
    mld_targets,
    selected_thresholds,
)

full_val_summary = pd.DataFrame({
    "Metric": [
        "Standard Macro-F1",
        "Official proxy F1",
        "Precision",
        "Recall",
        "Mean threshold",
        "Predicted labels / image",
    ],
    "Value": [
        selected_standard["macro_f1"],
        selected_proxy["f1"],
        selected_proxy["precision"],
        selected_proxy["recall"],
        selected_thresholds.mean().item(),
        (
            selected_val_predictions.sum().item()
            / len(selected_val_predictions)
        ),
    ],
})

display(
    full_val_summary.style
    .format({"Value": "{:.4f}"})
    .hide(axis="index")
)

Metric,Value
Standard Macro-F1,0.8055
Official proxy F1,0.7647
Precision,0.8441
Recall,0.6990
Mean threshold,0.5201
Predicted labels / image,2.7835


As expected, calibrating and evaluating thresholds on the same validation set produces a substantially higher score (**0.7647**) than the out-of-fold estimate (**0.7500**).

I therefore use **0.7500**, not 0.7647, as the more realistic local estimate of this candidate.

## 9. Hidden-test evaluation

After selecting the ensemble weight with out-of-fold validation, I apply the thresholds calibrated on the complete validation set to the hidden test set.

The hidden labels are never used during model selection. The test submission therefore uses the configuration selected exclusively from the validation experiments:

- **60% reference ensemble**
- **40% ML-Decoder**
- per-class thresholds calibrated on the complete validation set

In [22]:
MLD_TEST_CACHE = (
    CHECKPOINT_DIR
    / "convnext_base_mldecoder_asl_tta_test_probs.pt"
)

BASELINE_TEST_CACHE = (
    CHECKPOINT_DIR
    / "convnext_vit_resnet_multiview_test_probs.pt"
)

mld_test = torch.load(
    MLD_TEST_CACHE,
    map_location="cpu",
    weights_only=False,
)

baseline_test = torch.load(
    BASELINE_TEST_CACHE,
    map_location="cpu",
    weights_only=False,
)

mld_test_probs = mld_test["probs"]
mld_test_ids = list(mld_test["ids"])

baseline_test_probs = baseline_test["probs"]
baseline_test_ids = list(baseline_test["ids"])

assert mld_test_probs.shape == baseline_test_probs.shape
assert mld_test_ids == baseline_test_ids

test_probs = (
    SELECTED_BASELINE_WEIGHT * baseline_test_probs
    + SELECTED_MLD_WEIGHT * mld_test_probs
)

test_predictions = (
    test_probs
    > selected_thresholds.unsqueeze(0)
)

submission = {
    image_id: torch.where(prediction)[0].tolist()
    for image_id, prediction in zip(
        baseline_test_ids,
        test_predictions,
    )
}

SUBMISSION_FILE = (
    SUBMISSION_DIR
    / "submission_baseline_plus_mldecoder.json"
)

with open(SUBMISSION_FILE, "w") as f:
    json.dump(submission, f, indent=4)

print("Test images:", len(submission))
print(
    "Predicted labels / image:",
    round(
        np.mean([
            len(labels)
            for labels in submission.values()
        ]),
        3,
    ),
)

Test images: 4952
Predicted labels / image: 2.799


### 9.1 Hidden-test result

The selected ensemble was then evaluated on the hidden test set.

| Metric | Hidden test |
|---|---:|
| Accuracy | 0.7850 |
| F1 | **0.6863** |
| Precision | 0.7583 |
| Recall | 0.6268 |

Despite reaching **0.7500** with out-of-fold validation, the model obtains only **0.6863** on the hidden test set.

### 9.2 Validation–test gap

This result does not confirm the improvement observed locally.

The reference ensemble achieved:

- **0.7297** OOF proxy F1
- **0.7018** hidden-test F1

After adding 40% ML-Decoder, the scores became:

- **0.7500** OOF proxy F1
- **0.6863** hidden-test F1

Therefore, the local OOF score increased by approximately **0.0202**, while the hidden-test F1 decreased by approximately **0.0155**.

This result is consistent with the instability already visible during cross-validation. The ML-Decoder mixture had a substantially larger fold standard deviation and a worse minimum-fold score than the reference ensemble.

I therefore conclude that the 40% ML-Decoder mixture overfits the validation distribution despite its higher aggregate OOF score. I do not retain it as my final model.